# 02 — Data Quality & Cleaning (PySpark)

## Objective
Identify and fix all data quality issues across all five
tables using PySpark distributed operations — no pandas,
no row-by-row loops, no .collect() on large DataFrames.

## What We Fix
- Duplicate rows
- Impossible values (age, income, amounts)
- Date string → proper DateType casting
- Null imputation strategies per column
- Outlier capping using percentile bounds
- Standardizing categorical values (city names, segments)

## New PySpark Concepts in This Notebook
- F.when().otherwise() for conditional imputation
- F.percentile_approx() for outlier bounds inside groupBy
- Window functions for forward-fill within customer series
- .fillna() with different values per column
- Writing cleaned data to Parquet (faster than CSV for Spark)

## The Golden Rule
Never call .collect() or .toPandas() on a large DataFrame.
Only call these after aggregation when result is small.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    DateType, BooleanType, DoubleType,
    IntegerType, StringType
)
import os

JAR_PATH = os.path.abspath("../jars/sqlite-jdbc-3.45.1.0.jar")
DB_PATH  = r"D:\some\other\drive\finance_clv.db"
DB_URL   = f"jdbc:sqlite:{DB_PATH}"

spark = (
    SparkSession.builder
    .appName("CLV_Data_Quality")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.extraClassPath", JAR_PATH)
    .config("spark.executor.extraClassPath", JAR_PATH)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def read_table(name):
    return (
        spark.read.format("jdbc")
        .option("url", DB_URL)
        .option("dbtable", name)
        .option("driver", "org.sqlite.JDBC")
        .load()
    )

customers   = read_table("customers")
products    = read_table("products")
txn_history = read_table("transactions_history")
clv_labels  = read_table("clv_labels")

print("All tables loaded ✅")
print(f"  customers   : {customers.count():,}")
print(f"  products    : {products.count():,}")
print(f"  txn_history : {txn_history.count():,}")
print(f"  clv_labels  : {clv_labels.count():,}")

In [ ]:
print("STEP 1 — REMOVE DUPLICATES\n")

before = {
    "customers"  : customers.count(),
    "products"   : products.count(),
    "txn_history": txn_history.count(),
    "clv_labels" : clv_labels.count(),
}

# Remove exact duplicate rows first
customers   = customers.dropDuplicates()
products    = products.dropDuplicates()
txn_history = txn_history.dropDuplicates()
clv_labels  = clv_labels.dropDuplicates()

# Remove primary key duplicates — keep first occurrence
customers   = customers.dropDuplicates(["customer_id"])
products    = products.dropDuplicates(["product_id"])
txn_history = txn_history.dropDuplicates(["txn_id"])
clv_labels  = clv_labels.dropDuplicates(["customer_id"])

after = {
    "customers"  : customers.count(),
    "products"   : products.count(),
    "txn_history": txn_history.count(),
    "clv_labels" : clv_labels.count(),
}

print(f"{'Table':<15} {'Before':>10} {'After':>10} {'Removed':>10}")
print("-" * 48)
for name in before:
    removed = before[name] - after[name]
    print(f"  {name:<13} {before[name]:>10,} "
          f"{after[name]:>10,} {removed:>10,}")

In [ ]:
print("STEP 2 — CAST DATE COLUMNS\n")

# Fix customers
customers = customers.withColumn(
    "account_open_date",
    F.to_date(F.col("account_open_date"), "yyyy-MM-dd")
)

# Fix products
products = products.withColumn(
    "open_date",
    F.to_date(F.col("open_date"), "yyyy-MM-dd")
)

# Fix transactions — also extract year, month, day
txn_history = txn_history.withColumn(
    "txn_date",
    F.to_date(F.col("txn_date"), "yyyy-MM-dd")
).withColumn(
    "txn_year",  F.year("txn_date")
).withColumn(
    "txn_month", F.month("txn_date")
).withColumn(
    "txn_day",   F.dayofmonth("txn_date")
).withColumn(
    "txn_dayofweek", F.dayofweek("txn_date")
).withColumn(
    "is_weekend",
    (F.dayofweek("txn_date").isin([1, 7])).cast(IntegerType())
)

# Verify — count null dates after conversion
# If format didn't match, to_date returns null silently
null_dates = txn_history.filter(
    F.col("txn_date").isNull()
).count()

print(f"  Null txn_date after conversion : {null_dates:,}")
print(f"  (should be 0 — format must match)")

print("\nSample with new date columns:")
txn_history.select(
    "txn_id", "txn_date", "txn_year",
    "txn_month", "txn_dayofweek", "is_weekend"
).show(5)

In [ ]:
print("STEP 3 — FIX BOOLEAN AND INTEGER COLUMNS\n")

# Customers — convert 0/1 integers to proper boolean
bool_cols_customers = [
    "kyc_complete", "pan_linked", "aadhaar_linked",
    "rm_assigned", "is_nri"
]
for col in bool_cols_customers:
    customers = customers.withColumn(
        col, F.col(col).cast(BooleanType())
    )

# Products
products = products.withColumn(
    "is_active", F.col("is_active").cast(BooleanType())
)

# Transactions
txn_history = txn_history.withColumn(
    "is_international",
    F.col("is_international").cast(BooleanType())
)

# Verify schema after casting
print("Customers schema (key columns):")
customers.select(
    "customer_id", "kyc_complete", "pan_linked",
    "rm_assigned", "monthly_income", "cibil_score"
).printSchema()

print("Boolean value distribution (customers):")
customers.select(
    F.sum(F.col("kyc_complete").cast(IntegerType())).alias("kyc_complete_count"),
    F.sum(F.col("pan_linked").cast(IntegerType())).alias("pan_linked_count"),
    F.sum(F.col("rm_assigned").cast(IntegerType())).alias("rm_assigned_count"),
    F.count("customer_id").alias("total_customers")
).show()

In [ ]:
print("STEP 4 — FIX IMPOSSIBLE VALUES\n")

# ── Customers ─────────────────────────────────────────────────
print("── Customers ──")

# Age: must be 18-100
impossible_age = customers.filter(
    (F.col("age") < 18) | (F.col("age") > 100)
).count()
print(f"  Impossible age values   : {impossible_age:,}")

customers = customers.withColumn(
    "age",
    F.when(
        (F.col("age") < 18) | (F.col("age") > 100), None
    ).otherwise(F.col("age"))
)

# CIBIL score: must be 300-900 or null (no credit history)
impossible_cibil = customers.filter(
    F.col("cibil_score").isNotNull() &
    ((F.col("cibil_score") < 300) | (F.col("cibil_score") > 900))
).count()
print(f"  Impossible CIBIL values : {impossible_cibil:,}")

customers = customers.withColumn(
    "cibil_score",
    F.when(
        F.col("cibil_score").isNotNull() &
        ((F.col("cibil_score") < 300) | (F.col("cibil_score") > 900)),
        None
    ).otherwise(F.col("cibil_score"))
)

# Monthly income: must be positive
customers = customers.withColumn(
    "monthly_income",
    F.when(F.col("monthly_income") <= 0, None)
     .otherwise(F.col("monthly_income"))
)

# ── Transactions ───────────────────────────────────────────────
print("\n── Transactions ──")

# Amount: must be positive
neg_amounts = txn_history.filter(F.col("amount") <= 0).count()
print(f"  Negative/zero amounts   : {neg_amounts:,}")

txn_history = txn_history.filter(F.col("amount") > 0)

# Balance after cannot be negative (floor at 0)
txn_history = txn_history.withColumn(
    "balance_after",
    F.when(F.col("balance_after") < 0, 0.0)
     .otherwise(F.col("balance_after"))
)

print("\nAfter fixing impossible values:")
print(f"  txn_history rows: {txn_history.count():,}")

In [ ]:
print("STEP 5 — NULL IMPUTATION\n")

# ── Customers ─────────────────────────────────────────────────
print("── Customers ──")

# Age → median by segment
age_medians = customers.groupBy("segment").agg(
    F.percentile_approx("age", 0.5, 1000).alias("median_age")
)
print("  Median age by segment:")
age_medians.show()

customers = customers.join(
    age_medians, on="segment", how="left"
).withColumn(
    "age",
    F.coalesce(F.col("age"), F.col("median_age"))
).drop("median_age")

# CIBIL score → median by segment
cibil_medians = customers.groupBy("segment").agg(
    F.percentile_approx("cibil_score", 0.5, 1000).alias("median_cibil")
)
customers = customers.join(
    cibil_medians, on="segment", how="left"
).withColumn(
    "cibil_score",
    F.coalesce(F.col("cibil_score"), F.col("median_cibil"))
).drop("median_cibil")

# Gender → "Not Specified"
customers = customers.fillna(
    {"gender": "Not Specified"}
)

# Monthly income → median by segment (for the rare null)
income_medians = customers.groupBy("segment").agg(
    F.percentile_approx("monthly_income", 0.5, 1000).alias("median_income")
)
customers = customers.join(
    income_medians, on="segment", how="left"
).withColumn(
    "monthly_income",
    F.coalesce(F.col("monthly_income"), F.col("median_income"))
).drop("median_income")

# ── Transactions ───────────────────────────────────────────────
print("── Transactions ──")

# merchant_category and channel → "unknown"
txn_history = txn_history.fillna({
    "merchant_category": "unknown",
    "channel"          : "unknown",
})

# Verify
remaining_nulls = txn_history.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ["merchant_category", "channel", "amount"]
]).collect()[0]

print(f"\n  merchant_category nulls : {remaining_nulls['merchant_category']}")
print(f"  channel nulls           : {remaining_nulls['channel']}")
print(f"  amount nulls            : {remaining_nulls['amount']}")
print(f"  (all should be 0)")

In [ ]:
print("STEP 6 — OUTLIER CAPPING\n")

# ── Transaction amounts — cap at 99.9th percentile ────────────
print("── Transaction Amount Capping ──")
quantiles = txn_history.approxQuantile(
    "amount", [0.001, 0.999], relativeError=0.01
)
lower_bound = quantiles[0]
upper_bound = quantiles[1]

print(f"  Lower bound (P0.1)  : ₹{lower_bound:>12,.2f}")
print(f"  Upper bound (P99.9) : ₹{upper_bound:>12,.2f}")

before_cap = txn_history.count()
txn_history = txn_history.withColumn(
    "amount",
    F.when(F.col("amount") < lower_bound, lower_bound)
     .when(F.col("amount") > upper_bound, upper_bound)
     .otherwise(F.col("amount"))
)
print(f"  Rows before/after capping : {before_cap:,} / {txn_history.count():,}")
print(f"  (should be same — we cap, not drop)")

# ── Customer income — cap at 99th percentile ──────────────────
print("\n── Monthly Income Capping ──")
income_quantiles = customers.approxQuantile(
    "monthly_income", [0.01, 0.99], relativeError=0.01
)
inc_lower = income_quantiles[0]
inc_upper = income_quantiles[1]

print(f"  Lower bound (P1)  : ₹{inc_lower:>12,.2f}")
print(f"  Upper bound (P99) : ₹{inc_upper:>12,.2f}")

customers = customers.withColumn(
    "monthly_income",
    F.when(F.col("monthly_income") < inc_lower, inc_lower)
     .when(F.col("monthly_income") > inc_upper, inc_upper)
     .otherwise(F.col("monthly_income"))
)

print("\nIncome distribution after capping:")
customers.select(
    F.min("monthly_income").alias("min"),
    F.percentile_approx("monthly_income", 0.25, 1000).alias("p25"),
    F.percentile_approx("monthly_income", 0.50, 1000).alias("median"),
    F.percentile_approx("monthly_income", 0.75, 1000).alias("p75"),
    F.max("monthly_income").alias("max")
).show()

In [ ]:
print("STEP 7 — ADD DERIVED COLUMNS AND SAVE\n")

# ── Add account age in months (as of history period end) ──────
customers = customers.withColumn(
    "account_age_months",
    F.round(
        F.months_between(
            F.lit("2023-12-31").cast(DateType()),
            F.col("account_open_date")
        ), 1
    )
)

# ── Add income band ───────────────────────────────────────────
customers = customers.withColumn(
    "income_band",
    F.when(F.col("monthly_income") <  15000,  "very_low")
     .when(F.col("monthly_income") <  40000,  "low")
     .when(F.col("monthly_income") < 100000,  "mid")
     .when(F.col("monthly_income") < 300000,  "high")
     .otherwise("very_high")
)

# ── Verify final null counts ───────────────────────────────────
print("Final null counts — customers:")
customers.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ["age","gender","monthly_income","cibil_score",
              "account_open_date","account_age_months"]
]).show()

# ── Save to Parquet ───────────────────────────────────────────
PROCESSED_PATH = "../data/processed"

print("Saving cleaned data to Parquet...")

customers.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/customers_clean"
)
products.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/products_clean"
)
txn_history.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/transactions_clean"
)
clv_labels.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/clv_labels_clean"
)

print("✅ All tables saved to Parquet")

# Verify saved files can be read back
customers_check = spark.read.parquet(
    f"{PROCESSED_PATH}/customers_clean"
)
print(f"\nVerification — customers_clean rows: "
      f"{customers_check.count():,}")
print(f"Columns: {len(customers_check.columns)}")